Bill of Quantities - PV Paiva (DuckDB version)
=
This notebook measures the AC/DC trenches, ducts and cables drawn in the PV Paiva BOQ drawing and writes the quantities to `C:\Users\Usuario\Desktop\BOQ.xlsx`. It is the DuckDB SQL version of `BOQ.ipynb`: the groups are selected with `LIKE` patterns on the layer name, and summed/counted with SQL.

**Workflow overview:**
1. Convert the DWG to DXF and load all model space entities into `pv_projects.duckdb` (project `"PV Paiva"`).
2. Select the geometry of the project as a DuckDB relation (`cable_data`).
3. Split it into six groups by layer name: AC trenches, AC watercourse ductbanks, AC access ductbanks, DC cables, DC ducts, DC trenches.
4. For each group: list the entities, sum `length` per layer and, for the ducts, count the pieces per layer.
5. Write every summary table to its own sheet in `BOQ.xlsx` and close the connection.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `PV Paiva - BOQ.dwg` in `C:\Users\Usuario\Desktop\projetos\paiva`. The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** only model space entities are read. Geometry inside blocks, xrefs or paper space isn't counted.
- **Entity types:** every run must be a `LINE` or `LWPOLYLINE`, as only these get a `length`. Arcs, splines, old-style `POLYLINE`s, etc. are read without a length and add nothing.
- **Units:** drawing units are metres (the results are reported as m).
- **Layer names** (SQL `LIKE`, case-sensitive; `%` = any text, and `_` also matches any single character):
  - AC trenches: `EMF_bt_vala_btca_<n>xC`.
  - AC ducts in watercourse crossings: `EMF_bt_watercourse_ductbank_<n>xC`.
  - AC ducts in access crossings: `EMF_bt_access_ductbank_<n>xC`.
  - DC cables: exactly `EMF_cabo_solar_negativo` and `EMF_cabo_solar_positivo`.
  - DC ducts: `EMF_bt_eletroduto_<n>S`.
  - DC trenches: `EMF_vala_CC_<n>C`.
  - Entities on any other layer are ignored.
- **Ducts and crossings are counted per entity:** the unit count is the number of entities on the layer, so each duct piece / crossing must be one single LINE or LWPOLYLINE (not exploded into segments, not joined with the next one).

Extracting the AutoCAD data and loading it into the DuckDB database
=
- Copies `PV Paiva - BOQ.dwg` from `dwg_folder` into `temp_folder`, so the ODA File Converter only converts that one file and not every DWG in the project folder.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) writes the DXF (ACAD2018) into `output_folder` (`...\paiva\output`, a different folder from the DWG).
- `load_to_duckdb` reads every model space entity with `ezdxf` and loads it into the `cad_entities` table under the project name `"PV Paiva"`. If the project already exists, its old rows are deleted first.
  - `LINE`: start/end point and straight length.
  - `LWPOLYLINE`: first/last vertex and the length along all segments (bulge arcs included, plus the closing segment if the polyline is closed).
- At the end, the temporary DXF and the copied DWG are deleted. The DuckDB connection (`conn`) stays open for the next cell.
- **Warning:** `Tagueamento Eletromecânico.ipynb` also loads its drawing under the project name `"PV Paiva"`. Whichever notebook runs last replaces the other one's rows in `cad_entities`.

In [83]:
from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - BOQ.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - BOQ.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - BOQ.dwg")

conn = load_to_duckdb(dxf_file, "PV Paiva")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

1
Project PV Paiva already exists
Removed existing data for project: PV Paiva
Successfully loaded project
Loaded 40737 entities for project: PV Paiva
Don't forget to close connection.
Data Loaded
DuckDB Loaded


Selecting the cable data
=
- `cable_data` is a DuckDB relation with `layer`, start/end coordinates and `length` of project `"PV Paiva"`. The next cells query it by its Python variable name (DuckDB can query Python relations/DataFrames by name).
- `DESCRIBE cad_entities` prints the table schema.
- The layer count runs on `cad_entities` **without a project filter**, so it counts the layers of every project in the database, not only PV Paiva. Query `cable_data` instead to see only this drawing.

In [84]:
cable_data = conn.sql("SELECT layer, start_x, start_y, end_x, end_y, length FROM cad_entities WHERE project_name='PV Paiva'")
print(cable_data)
print(conn.execute("DESCRIBE cad_entities").fetchdf())

result = conn.execute("""
    SELECT
        layer,
        COUNT(*) as count
    FROM cad_entities
    GROUP BY layer
    ORDER BY count DESC
""").fetchdf()
print(result)

┌─────────────────┬───────────┬────────────┬────────────┬────────────┬───────────┐
│      layer      │  start_x  │  start_y   │   end_x    │   end_y    │  length   │
│     varchar     │   float   │   float    │   float    │   float    │   float   │
├─────────────────┼───────────┼────────────┼────────────┼────────────┼───────────┤
│ EMF_vala_CC_2C  │ 34683.586 │  126169.99 │   34685.28 │  126171.42 │ 2.2149422 │
│ EMF_vala_CC_4C  │  34685.28 │  126171.42 │  34689.887 │  126171.14 │   4.61177 │
│ EMF_vala_CC_1C  │ 34757.492 │  126242.08 │  34752.887 │  126242.08 │   4.60705 │
│ EMF_vala_CC_3C  │ 34536.992 │  126293.21 │  34533.984 │  126293.21 │  3.007685 │
│ EMF_vala_CC_2C  │ 34798.684 │  126336.31 │  34804.984 │  126336.31 │       6.3 │
│ EMF_vala_CC_3C  │ 34804.984 │  126336.31 │  34809.586 │ 126336.305 │  4.603367 │
│ EMF_vala_CC_8C  │  35157.78 │  126774.16 │  35162.387 │  126774.17 │ 4.6032186 │
│ EMF_vala_CC_10C │ 35740.277 │  127982.76 │ 35735.6875 │  127982.76 │  4.592198 │
│ EM

Splitting the entities into quantity groups
=
One relation per group, selected by layer name:
- `ac_cables`: `LIKE 'EMF_bt_vala_btca_%xC'` (AC trenches).
- `ac_watercourse`: `LIKE 'EMF_bt_watercourse_ductbank_%xC'`.
- `ac_access`: `LIKE 'EMF_bt_access_ductbank_%xC'`.
- `dc_cables`: exact match on the positive and negative solar cable layers.
- `dc_ducts`: `LIKE 'EMF_bt_eletroduto_%S'`.
- `dc_trenches`: `LIKE 'EMF_vala_CC_%C'`.

`LIKE` is case-sensitive. Unlike `BOQ.ipynb`, new circuit counts (e.g. `09xC`) are picked up automatically, and layers that match no pattern are left out instead of being counted as DC trenches. The relations are lazy: they are evaluated each time they are printed or queried.

In [85]:
ac_cables = conn.sql("""
    SELECT * FROM cable_data WHERE layer LIKE 'EMF_bt_vala_btca_%xC'
""")

ac_watercourse = conn.sql("""
    SELECT * FROM cable_data WHERE layer LIKE 'EMF_bt_watercourse_ductbank_%xC'
""")

ac_access = conn.sql("""
    SELECT * FROM cable_data WHERE layer LIKE 'EMF_bt_access_ductbank_%xC'
""")

dc_cables = conn.sql("""
    SELECT * FROM cable_data WHERE layer = 'EMF_cabo_solar_negativo' OR layer = 'EMF_cabo_solar_positivo'
""")

dc_ducts = conn.sql("""
    SELECT * FROM cable_data WHERE layer LIKE 'EMF_bt_eletroduto_%S'
""")

dc_trenches = conn.sql("""
    SELECT * FROM cable_data WHERE layer LIKE 'EMF_vala_CC_%C'
""")

QTY. OF AC CABLES (m)
=

Prints the AC trench entities (`vala` = trench). The `NNxC` suffix of the layer is the number of AC circuits in the trench.

In [86]:
print(ac_cables)

┌───────────────────────┬───────────┬───────────┬───────────┬────────────┬────────────┐
│         layer         │  start_x  │  start_y  │   end_x   │   end_y    │   length   │
│        varchar        │   float   │   float   │   float   │   float    │   float    │
├───────────────────────┼───────────┼───────────┼───────────┼────────────┼────────────┤
│ EMF_bt_vala_btca_01xC │  36054.69 │ 129707.17 │ 35947.617 │  129701.53 │ 107.897514 │
│ EMF_bt_vala_btca_01xC │ 33680.184 │ 129012.41 │ 33680.184 │  129010.91 │        1.5 │
│ EMF_bt_vala_btca_01xC │ 33699.043 │ 128996.22 │ 33699.066 │  128997.72 │  1.5002186 │
│ EMF_bt_vala_btca_01xC │ 33730.543 │ 128996.22 │ 33730.566 │  128997.72 │  1.5002186 │
│ EMF_bt_vala_btca_01xC │ 33768.367 │ 128996.22 │ 33768.367 │  128997.72 │        1.5 │
│ EMF_bt_vala_btca_01xC │  33812.44 │ 128996.22 │  33812.47 │  128997.72 │  1.5002186 │
│ EMF_bt_vala_btca_01xC │ 33869.168 │ 128910.11 │ 33869.168 │  128912.58 │  2.4706268 │
│ EMF_bt_vala_btca_01xC │ 33913.

AC trench length per layer
=
`SUM(length)` per layer, sorted by layer name. `ac_cables_group` goes to the `AC Trenches (m)` sheet. The relation is referenced by its variable name (here written as the string `'ac_cables'`).

In [87]:
ac_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_cables' GROUP BY layer ORDER BY layer")
print(ac_cables_group)

┌───────────────────────┬────────────────────┐
│         layer         │    total_length    │
│        varchar        │       double       │
├───────────────────────┼────────────────────┤
│ EMF_bt_vala_btca_01xC │ 11612.351512610912 │
│ EMF_bt_vala_btca_02xC │  7535.069041728973 │
│ EMF_bt_vala_btca_03xC │ 5263.3760442733765 │
│ EMF_bt_vala_btca_04xC │  4043.333034515381 │
│ EMF_bt_vala_btca_05xC │  3160.560396671295 │
│ EMF_bt_vala_btca_06xC │  2192.055307865143 │
│ EMF_bt_vala_btca_07xC │ 1411.1449394226074 │
│ EMF_bt_vala_btca_08xC │  1724.205687046051 │
└───────────────────────┴────────────────────┘



QTY. OF AC DUCTS IN WATERCOURSE (m and un.)
=

Prints the ductbanks where the AC trenches cross a watercourse.

In [88]:
print(ac_watercourse)

┌──────────────────────────────────┬────────────┬────────────┬───────────┬────────────┬───────────┐
│              layer               │  start_x   │  start_y   │   end_x   │   end_y    │  length   │
│             varchar              │   float    │   float    │   float   │   float    │   float   │
├──────────────────────────────────┼────────────┼────────────┼───────────┼────────────┼───────────┤
│ EMF_bt_watercourse_ductbank_05xC │   33898.32 │  128997.72 │ 33875.418 │  128997.72 │  22.90354 │
│ EMF_bt_watercourse_ductbank_03xC │ 34134.8125 │  128777.04 │ 34155.098 │  128768.53 │ 21.999958 │
│ EMF_bt_watercourse_ductbank_02xC │   34118.69 │  128737.58 │ 34140.695 │  128737.58 │ 22.005062 │
│ EMF_bt_watercourse_ductbank_02xC │   35990.72 │  128880.65 │ 36012.758 │  128880.65 │ 22.037695 │
│ EMF_bt_watercourse_ductbank_04xC │  35750.203 │  129466.68 │ 35728.496 │  129470.25 │      22.0 │
│ EMF_bt_watercourse_ductbank_07xC │  34951.957 │ 129537.664 │  34972.07 │  129528.74 │ 22.004562 │


Watercourse ductbank length per layer
=
`SUM(length)` per layer. Goes to the `AC Ducts Watercouse (m)` sheet.

In [89]:
ac_watercourse_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_watercourse' GROUP BY layer ORDER BY layer")
print(ac_watercourse_group)

┌──────────────────────────────────┬────────────────────┐
│              layer               │    total_length    │
│             varchar              │       double       │
├──────────────────────────────────┼────────────────────┤
│ EMF_bt_watercourse_ductbank_01xC │  65.98100090026855 │
│ EMF_bt_watercourse_ductbank_02xC │  44.04275703430176 │
│ EMF_bt_watercourse_ductbank_03xC │ 44.005550384521484 │
│ EMF_bt_watercourse_ductbank_04xC │               44.0 │
│ EMF_bt_watercourse_ductbank_05xC │ 22.903539657592773 │
│ EMF_bt_watercourse_ductbank_07xC │  44.00456237792969 │
└──────────────────────────────────┴────────────────────┘



Watercourse ductbanks: number of units
=
`COUNT(layer)` per layer, sorted by count. Each entity is one crossing. Goes to the `AC Ducts Watercouse (un.)` sheet.

In [90]:
ac_watercourse_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'ac_watercourse' GROUP BY layer ORDER BY count DESC")
print(ac_watercourse_count)

┌──────────────────────────────────┬───────┐
│              layer               │ count │
│             varchar              │ int64 │
├──────────────────────────────────┼───────┤
│ EMF_bt_watercourse_ductbank_01xC │     3 │
│ EMF_bt_watercourse_ductbank_07xC │     2 │
│ EMF_bt_watercourse_ductbank_03xC │     2 │
│ EMF_bt_watercourse_ductbank_04xC │     2 │
│ EMF_bt_watercourse_ductbank_02xC │     2 │
│ EMF_bt_watercourse_ductbank_05xC │     1 │
└──────────────────────────────────┴───────┘



QTY. OF AC DUCTS IN ACCESS (m and un.)
=

Prints the ductbanks where the AC trenches cross an access road.

In [91]:
print(ac_access)

┌─────────────────────────────┬───────────┬────────────┬───────────┬────────────┬───────────┐
│            layer            │  start_x  │  start_y   │   end_x   │   end_y    │  length   │
│           varchar           │   float   │   float    │   float   │   float    │   float   │
├─────────────────────────────┼───────────┼────────────┼───────────┼────────────┼───────────┤
│ EMF_bt_access_ductbank_07xC │ 33777.883 │  129007.32 │ 33777.883 │  129001.32 │       6.0 │
│ EMF_bt_access_ductbank_02xC │ 34572.293 │  129115.38 │  34572.29 │  129109.38 │ 5.9999995 │
│ EMF_bt_access_ductbank_01xC │  34614.23 │  128840.96 │  34608.23 │  128840.96 │       6.0 │
│ EMF_bt_access_ductbank_02xC │ 34604.305 │  128663.77 │ 34618.152 │  128663.77 │ 13.848897 │
│ EMF_bt_access_ductbank_04xC │   34619.2 │  128646.43 │   34603.0 │  128646.43 │ 16.202538 │
│ EMF_bt_access_ductbank_01xC │  33985.89 │  128997.72 │  34005.11 │  128997.72 │ 19.217321 │
│ EMF_bt_access_ductbank_05xC │ 34614.227 │ 128511.875 │ 346

Access ductbank length per layer
=
`SUM(length)` per layer. Goes to the `AC Ducts Access (m)` sheet.

In [92]:
ac_access_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_access' GROUP BY layer ORDER BY layer")
print(ac_access_group)

┌─────────────────────────────┬────────────────────┐
│            layer            │    total_length    │
│           varchar           │       double       │
├─────────────────────────────┼────────────────────┤
│ EMF_bt_access_ductbank_01xC │  75.32514429092407 │
│ EMF_bt_access_ductbank_02xC │   45.6199631690979 │
│ EMF_bt_access_ductbank_03xC │ 31.175094604492188 │
│ EMF_bt_access_ductbank_04xC │ 28.202537536621094 │
│ EMF_bt_access_ductbank_05xC │  42.00000619888306 │
│ EMF_bt_access_ductbank_06xC │ 38.730658531188965 │
│ EMF_bt_access_ductbank_07xC │   27.3529691696167 │
│ EMF_bt_access_ductbank_08xC │  36.36680316925049 │
└─────────────────────────────┴────────────────────┘



Access ductbanks: number of units
=
`COUNT(layer)` per layer (one entity = one crossing). Goes to the `AC Ducts Access (un.)` sheet.

In [93]:
ac_access_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'ac_access' GROUP BY layer ORDER BY count DESC")
print(ac_access_count)

┌─────────────────────────────┬───────┐
│            layer            │ count │
│           varchar           │ int64 │
├─────────────────────────────┼───────┤
│ EMF_bt_access_ductbank_01xC │     8 │
│ EMF_bt_access_ductbank_05xC │     7 │
│ EMF_bt_access_ductbank_03xC │     5 │
│ EMF_bt_access_ductbank_08xC │     5 │
│ EMF_bt_access_ductbank_02xC │     5 │
│ EMF_bt_access_ductbank_06xC │     4 │
│ EMF_bt_access_ductbank_07xC │     4 │
│ EMF_bt_access_ductbank_04xC │     3 │
└─────────────────────────────┴───────┘



QTY. OF DC CABLES (m)
=

Prints every positive and negative solar cable run.

In [94]:
print(dc_cables)

┌─────────────────────────┬───────────┬────────────┬──────────┬───────────┬────────────┐
│          layer          │  start_x  │  start_y   │  end_x   │   end_y   │   length   │
│         varchar         │   float   │   float    │  float   │   float   │   float    │
├─────────────────────────┼───────────┼────────────┼──────────┼───────────┼────────────┤
│ EMF_cabo_solar_positivo │ 36016.906 │  129793.61 │ 36054.17 │ 129708.74 │  121.26314 │
│ EMF_cabo_solar_positivo │ 36016.906 │  129751.11 │ 36054.17 │ 129708.74 │   78.76113 │
│ EMF_cabo_solar_positivo │ 36023.207 │  129793.61 │ 36054.17 │ 129708.74 │ 114.963135 │
│ EMF_cabo_solar_positivo │ 36023.207 │  129751.11 │ 36054.17 │ 129708.74 │  72.461136 │
│ EMF_cabo_solar_positivo │ 36029.504 │  129793.61 │ 36054.17 │ 129708.74 │  108.66313 │
│ EMF_cabo_solar_positivo │ 36029.504 │  129751.11 │ 36054.17 │ 129708.74 │   66.16113 │
│ EMF_cabo_solar_positivo │ 36035.805 │  129793.61 │ 36054.17 │ 129708.74 │  102.36314 │
│ EMF_cabo_solar_posi

DC cable length per layer and total
=
- Per-layer `SUM(length)` for the positive and the negative layers.
- The grand total (both polarities) is a second query, rounded to 3 decimals. Only the per-layer table is exported (`DC Cables (m)`).

In [95]:
dc_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_cables' GROUP BY layer ORDER BY layer")
print(dc_cables_group)

print(f'The total length of DC cables is {round(conn.sql("SELECT SUM(length) FROM 'dc_cables'").fetchone()[0],3)} meters')

┌─────────────────────────┬────────────────────┐
│          layer          │    total_length    │
│         varchar         │       double       │
├─────────────────────────┼────────────────────┤
│ EMF_cabo_solar_negativo │ 1069015.7601782084 │
│ EMF_cabo_solar_positivo │  979820.5478003621 │
└─────────────────────────┴────────────────────┘

The total length of DC cables is 2048836.308 meters


QTY. OF DC DUCTS (m and un.)
=

Prints the DC duct entities.

In [96]:
print(dc_ducts)

┌──────────────────────┬────────────┬───────────┬───────────┬────────────┬───────────┐
│        layer         │  start_x   │  start_y  │   end_x   │   end_y    │  length   │
│       varchar        │   float    │   float   │   float   │   float    │   float   │
├──────────────────────┼────────────┼───────────┼───────────┼────────────┼───────────┤
│ EMF_bt_eletroduto_2S │  36016.906 │ 129710.07 │  36054.17 │  129708.74 │ 37.725536 │
│ EMF_bt_eletroduto_2S │  36023.207 │ 129710.07 │  36054.17 │  129708.74 │ 31.425535 │
│ EMF_bt_eletroduto_2S │  36029.504 │ 129710.07 │  36054.17 │  129708.74 │ 25.125534 │
│ EMF_bt_eletroduto_2S │  36035.805 │ 129710.07 │  36054.17 │  129708.74 │ 18.825535 │
│ EMF_bt_eletroduto_2S │  36042.105 │ 129710.07 │  36054.17 │  129708.74 │ 12.525535 │
│ EMF_bt_eletroduto_2S │  36048.406 │ 129710.07 │  36054.17 │  129708.74 │ 6.2255344 │
│ EMF_bt_eletroduto_2S │  36054.707 │ 129710.07 │ 36054.707 │ 129708.945 │  1.128225 │
│ EMF_bt_eletroduto_2S │  36061.004 │ 12971

DC duct length per layer
=
Per-layer sum for `1S` / `2S` / `3S` (the suffix is the number of conduits/strings in the duct). Goes to the `DC Ducts (m)` sheet.

In [97]:
dc_ducts_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_ducts' GROUP BY layer ORDER BY layer")
print(dc_ducts_group)

┌──────────────────────┬────────────────────┐
│        layer         │    total_length    │
│       varchar        │       double       │
├──────────────────────┼────────────────────┤
│ EMF_bt_eletroduto_1S │  41748.20395702124 │
│ EMF_bt_eletroduto_2S │ 31690.800749063492 │
│ EMF_bt_eletroduto_3S │  36640.67237442732 │
└──────────────────────┴────────────────────┘



DC ducts: number of units
=
Number of duct entities per layer and in total. Each LINE/LWPOLYLINE on a duct layer counts as one duct. Goes to the `DC Ducts (un.)` sheet.

In [98]:
dc_ducts_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'dc_ducts' GROUP BY layer ORDER BY count DESC")
print(dc_ducts_count)

print(f'The total quantity of DC ducts is {conn.sql("SELECT COUNT(layer) FROM 'dc_ducts'").fetchone()[0]}')

┌──────────────────────┬───────┐
│        layer         │ count │
│       varchar        │ int64 │
├──────────────────────┼───────┤
│ EMF_bt_eletroduto_3S │  2748 │
│ EMF_bt_eletroduto_2S │  1378 │
│ EMF_bt_eletroduto_1S │  1368 │
└──────────────────────┴───────┘

The total quantity of DC ducts is 5494


QTY. OF DC TRENCHES (m)
=

Prints the DC trench entities.

In [99]:
print(dc_trenches)

┌─────────────────┬────────────┬───────────┬────────────┬────────────┬───────────┐
│      layer      │  start_x   │  start_y  │   end_x    │   end_y    │  length   │
│     varchar     │   float    │   float   │   float    │   float    │   float   │
├─────────────────┼────────────┼───────────┼────────────┼────────────┼───────────┤
│ EMF_vala_CC_2C  │  34683.586 │ 126169.99 │   34685.28 │  126171.42 │ 2.2149422 │
│ EMF_vala_CC_4C  │   34685.28 │ 126171.42 │  34689.887 │  126171.14 │   4.61177 │
│ EMF_vala_CC_1C  │  34757.492 │ 126242.08 │  34752.887 │  126242.08 │   4.60705 │
│ EMF_vala_CC_3C  │  34536.992 │ 126293.21 │  34533.984 │  126293.21 │  3.007685 │
│ EMF_vala_CC_2C  │  34798.684 │ 126336.31 │  34804.984 │  126336.31 │       6.3 │
│ EMF_vala_CC_3C  │  34804.984 │ 126336.31 │  34809.586 │ 126336.305 │  4.603367 │
│ EMF_vala_CC_8C  │   35157.78 │ 126774.16 │  35162.387 │  126774.17 │ 4.6032186 │
│ EMF_vala_CC_10C │  35740.277 │ 127982.76 │ 35735.6875 │  127982.76 │  4.592198 │
│ EM

DC trench length per layer and total
=
Per-layer sum of the `EMF_vala_CC_<n>C` layers (`<n>C` = number of DC circuits in the trench), plus the grand total. Goes to the sheet named `DC Trenches (un.)`, although the values are lengths in metres.

In [100]:
dc_trenches_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_trenches' GROUP BY layer ORDER BY layer")
print(dc_trenches_group)

print(f'\nThe total length of DC trenches is {round(conn.sql("SELECT SUM(length) FROM 'dc_trenches'").fetchone()[0],3)} meters')

┌─────────────────┬────────────────────┐
│      layer      │    total_length    │
│     varchar     │       double       │
├─────────────────┼────────────────────┤
│ EMF_vala_CC_10C │ 1440.6499154567719 │
│ EMF_vala_CC_11C │  970.9225063323975 │
│ EMF_vala_CC_12C │ 2135.3085206747055 │
│ EMF_vala_CC_13C │  583.5266884565353 │
│ EMF_vala_CC_14C │ 406.08687794208527 │
│ EMF_vala_CC_15C │ 265.98741441965103 │
│ EMF_vala_CC_16C │  52.20628476142883 │
│ EMF_vala_CC_17C │  64.54491519927979 │
│ EMF_vala_CC_18C │ 12.213818550109863 │
│ EMF_vala_CC_19C │  4.074550151824951 │
│ EMF_vala_CC_1C  │  3694.786339044571 │
│ EMF_vala_CC_20C │  6.300000190734863 │
│ EMF_vala_CC_21C │  4.074550151824951 │
│ EMF_vala_CC_23C │  4.074391841888428 │
│ EMF_vala_CC_2C  │ 5324.6826449632645 │
│ EMF_vala_CC_3C  │   7273.21101307869 │
│ EMF_vala_CC_4C  │ 4043.1264393925667 │
│ EMF_vala_CC_5C  │ 1963.3808864355087 │
│ EMF_vala_CC_6C  │  3588.523284316063 │
│ EMF_vala_CC_7C  │ 1465.7026493549347 │
│ EMF_vala_CC_8C

OVERVIEW
=

Writes every summary relation (`.df()` converts it to pandas) to its own sheet of `C:\Users\Usuario\Desktop\BOQ.xlsx`, then closes the DuckDB connection.
- Same output path as `BOQ.ipynb`: running either notebook overwrites the other one's file.
- The `DC Trenches (un.)` sheet holds lengths in metres, not units. `Watercouse` is misspelled in two sheet names.

In [101]:
with pd.ExcelWriter(r"C:\Users\Usuario\Desktop\BOQ.xlsx") as xlsx:
    ac_cables_group.df().to_excel(xlsx, sheet_name='AC Trenches (m)', index=False)
    ac_watercourse_group.df().to_excel(xlsx, sheet_name='AC Ducts Watercouse (m)', index=False)
    ac_watercourse_count.df().to_excel(xlsx, sheet_name='AC Ducts Watercouse (un.)', index=False)
    ac_access_group.df().to_excel(xlsx, sheet_name='AC Ducts Access (m)', index=False)
    ac_access_count.df().to_excel(xlsx, sheet_name='AC Ducts Access (un.)', index=False)
    dc_cables_group.df().to_excel(xlsx, sheet_name='DC Cables (m)', index=False)
    dc_ducts_group.df().to_excel(xlsx, sheet_name='DC Ducts (m)', index=False)
    dc_ducts_count.df().to_excel(xlsx, sheet_name='DC Ducts (un.)', index=False)
    dc_trenches_group.df().to_excel(xlsx, sheet_name='DC Trenches (un.)', index=False)


conn.close()


DDB process took 4.632062196731567 seconds
